# Sleepâ€“wake classification with sparse labels (simulation)

Hold out one quarter of participants. Train a 96-tree forest with 10% of the training epochs labeled. Retrain after adding training-only pseudo-labels with probability below 0.08 or above 0.92. Metrics pool held-out epochs within each split. Generate 40 people with person-specific heart-rate baselines and switching wake/sleep states. Compute rolling summaries from the synthetic signal.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'label-efficient-sleep-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    balanced_accuracy_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    rng = np.random.default_rng(seed)\n    x = []\n    y = []\n    g = []\n    for person in range(40):\n        wake = True\n        offset = rng.normal(0, 5)\n        history = []\n        for epoch in range(400):\n            if rng.random() < (0.07 if wake else 0.025):\n                wake = not wake\n            rate = 65 + offset + 7 * wake + rng.normal(0, 4)\n            history.append(rate)\n            window = np.array(history[-10:])\n            x.append(\n                [\n                    rate,\n                    window.mean(),\n                    window.std(),\n                    np.ptp(window),\n                    rate - window.mean(),\n                ]\n            )\n            y.append(int(not wake))\n            g.append(person)\n    return np.array(x), np.array(y), np.array(g)\n\n\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.model_selection import GroupShuffleSplit\n\n\ndef run(seed):\n    x, y, g = dataset(seed)\n    tr, te = next(\n        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(x, y, g)\n    )\n    rng = np.random.default_rng(seed)\n    labeled = rng.choice(tr, int(0.10 * len(tr)), replace=False)\n    unlabeled = np.setdiff1d(tr, labeled)\n\n    def fit(a, b):\n        return ExtraTreesClassifier(\n            n_estimators=96,\n            min_samples_leaf=8,\n            class_weight="balanced",\n            random_state=seed,\n            n_jobs=2,\n        ).fit(a, b)\n\n    initial = fit(x[labeled], y[labeled])\n    p = initial.predict_proba(x[unlabeled])[:, 1]\n    accepted = np.abs(p - 0.5) > 0.42\n    expanded = fit(\n        np.concatenate([x[labeled], x[unlabeled][accepted]]),\n        np.r_[y[labeled], p[accepted] >= 0.5],\n    )\n    output = {}\n    for name, model in [("labeled_only", initial), ("pseudo_labels", expanded)]:\n        prob = model.predict_proba(x[te])[:, 1]\n        output[name] = {\n            **scores(y[te], prob),\n            "balanced_accuracy": float(balanced_accuracy_score(y[te], prob >= 0.5)),\n            "test_subjects": len(np.unique(g[te])),\n        }\n    save((initial, expanded, np.unique(g[te]), int(accepted.sum())), f"models-{seed}")\n    return output\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())